# Healthcare Claims Analysis Project

This notebook combines and cleans your earlier SQL practice files into one **portfolio-ready project notebook**.

## Project Goals
- Build a small healthcare claims database in SQLite
- Analyze claims, members, and providers
- Create business-focused SQL insights
- Demonstrate joins, aggregations, CTEs, and window functions

## Tools Used
- SQL (SQLite)
- Python
- Pandas
- Jupyter Notebook


## 1. Import Libraries and Create Database Connection

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")
cur = con.cursor()

## 2. Create Sample Tables

We will use three tables:
- `members`
- `providers`
- `claims`


In [ ]:
cur.execute("DROP TABLE IF EXISTS claims")
cur.execute("DROP TABLE IF EXISTS members")
cur.execute("DROP TABLE IF EXISTS providers")

cur.execute("""
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    member_name TEXT,
    age INTEGER,
    gender TEXT
)
""")

cur.execute("""
CREATE TABLE providers (
    provider_id INTEGER PRIMARY KEY,
    provider_name TEXT,
    city TEXT
)
""")

cur.execute("""
CREATE TABLE claims (
    claim_id INTEGER PRIMARY KEY,
    member_id INTEGER,
    provider_id INTEGER,
    claim_date TEXT,
    claim_amount REAL,
    claim_status TEXT,
    FOREIGN KEY (member_id) REFERENCES members(member_id),
    FOREIGN KEY (provider_id) REFERENCES providers(provider_id)
)
""")

## 3. Insert Sample Data

In [ ]:
members_data = [
    (1, 'John Doe', 35, 'M'),
    (2, 'Jane Smith', 29, 'F'),
    (3, 'James Brown', 42, 'M'),
    (4, 'Julia Roberts', 31, 'F'),
    (5, 'Jack Wilson', 27, 'M')
]

providers_data = [
    (1, 'UHC', 'New York'),
    (2, 'Aetna', 'Chicago'),
    (3, 'Cigna', 'Dallas'),
    (4, 'Humana', 'Boston')
]

claims_data = [
    (101, 1, 1, '2025-09-28', 22000, 'Pending'),
    (102, 2, 1, '2025-10-05', 12000, 'Approved'),
    (103, 3, 2, '2025-10-12', 15000, 'Approved'),
    (104, 4, 3, '2025-10-18',  8000, 'Pending'),
    (105, 5, 4, '2025-10-22',  7000, 'Pending'),
    (106, 1, 2, '2025-10-28', 18000, 'Pending'),
    (107, 2, 3, '2025-11-01',  7000, 'Pending'),
    (108, 3, 1, '2025-11-04', 25000, 'Approved'),
    (109, 2, 1, '2025-11-05',  9000, 'Approved'),
    (110, 2, 1, '2025-11-09', 11000, 'Pending'),
    (111, 1, 2, '2025-11-15', 24000, 'Approved')
]

cur.executemany("INSERT INTO members VALUES (?, ?, ?, ?)", members_data)
cur.executemany("INSERT INTO providers VALUES (?, ?, ?)", providers_data)
cur.executemany("INSERT INTO claims VALUES (?, ?, ?, ?, ?, ?)", claims_data)
con.commit()

## 4. Preview the Data

In [ ]:
display(pd.read_sql_query("SELECT * FROM members", con))
display(pd.read_sql_query("SELECT * FROM providers", con))
display(pd.read_sql_query("SELECT * FROM claims ORDER BY claim_date", con))

## 5. Basic KPI Analysis

### Question 1
What is the total number of claims?


In [ ]:
query_1 = """
SELECT COUNT(*) AS total_claims
FROM claims
"""
pd.read_sql_query(query_1, con)

### Question 2
What is the total claim amount?


In [ ]:
query_2 = """
SELECT SUM(claim_amount) AS total_claim_amount
FROM claims
"""
pd.read_sql_query(query_2, con)

### Question 3
How many claims does each provider have?


In [ ]:
query_3 = """
SELECT provider_id,
       COUNT(*) AS claims_per_provider
FROM claims
GROUP BY provider_id
ORDER BY claims_per_provider DESC, provider_id
"""
pd.read_sql_query(query_3, con)

### Question 4
Show claim status distribution.


In [ ]:
query_4 = """
SELECT claim_status,
       COUNT(*) AS total_claims,
       SUM(claim_amount) AS total_amount
FROM claims
GROUP BY claim_status
ORDER BY total_claims DESC
"""
pd.read_sql_query(query_4, con)

## 6. Provider and Member Analysis

### Question 5
Find total billed amount by provider.


In [ ]:
query_5 = """
SELECT provider_id,
       SUM(claim_amount) AS total_claim_amount
FROM claims
GROUP BY provider_id
ORDER BY total_claim_amount DESC
"""
pd.read_sql_query(query_5, con)

### Question 6
Find total claim count by member.


In [ ]:
query_6 = """
SELECT member_id,
       COUNT(*) AS total_claims
FROM claims
GROUP BY member_id
ORDER BY total_claims DESC, member_id
"""
pd.read_sql_query(query_6, con)

### Question 7
Find the monthly claim trend.


In [ ]:
query_7 = """
SELECT strftime('%Y-%m', claim_date) AS claim_month,
       COUNT(*) AS total_claims,
       SUM(claim_amount) AS total_amount
FROM claims
GROUP BY strftime('%Y-%m', claim_date)
ORDER BY claim_month
"""
pd.read_sql_query(query_7, con)

## 7. Create a Master Dataset with Joins

### Question 8
Combine claims, members, and providers into a single business-friendly dataset.


In [ ]:
query_8 = """
SELECT c.claim_id,
       c.claim_date,
       c.claim_amount,
       c.claim_status,
       m.member_id,
       m.member_name,
       m.age,
       m.gender,
       p.provider_id,
       p.provider_name,
       p.city
FROM claims c
LEFT JOIN members m
    ON c.member_id = m.member_id
LEFT JOIN providers p
    ON c.provider_id = p.provider_id
ORDER BY c.claim_date
"""
pd.read_sql_query(query_8, con)

## 8. Business-Focused Analytical Queries

### Question 9
Find the highest claim amount for each provider.


In [ ]:
query_9 = """
SELECT provider_id,
       MAX(claim_amount) AS highest_claim_amount
FROM claims
GROUP BY provider_id
ORDER BY highest_claim_amount DESC
"""
pd.read_sql_query(query_9, con)

### Question 10
Find the member with the most approved claims.


In [ ]:
query_10 = """
SELECT member_id,
       COUNT(*) AS approved_claims
FROM claims
WHERE claim_status = 'Approved'
GROUP BY member_id
ORDER BY approved_claims DESC, member_id
"""
pd.read_sql_query(query_10, con)

### Question 11
Show members who have at least one approved claim.


In [ ]:
query_11 = """
SELECT m.member_id,
       m.member_name
FROM members m
WHERE EXISTS (
    SELECT 1
    FROM claims c
    WHERE c.member_id = m.member_id
      AND c.claim_status = 'Approved'
)
ORDER BY m.member_id
"""
pd.read_sql_query(query_11, con)

### Question 12
Show members who do not have any approved claim.


In [ ]:
query_12 = """
SELECT m.member_id,
       m.member_name
FROM members m
WHERE NOT EXISTS (
    SELECT 1
    FROM claims c
    WHERE c.member_id = m.member_id
      AND c.claim_status = 'Approved'
)
ORDER BY m.member_id
"""
pd.read_sql_query(query_12, con)

## 9. Fraud and Risk Flags

### Question 13
Flag claims above 20,000 as high-value claims.


In [ ]:
query_13 = """
SELECT claim_id,
       member_id,
       provider_id,
       claim_date,
       claim_amount,
       claim_status,
       CASE
           WHEN claim_amount > 20000 THEN 1
           ELSE 0
       END AS high_value_flag
FROM claims
ORDER BY claim_amount DESC
"""
pd.read_sql_query(query_13, con)

### Question 14
Flag same-member, same-day multiple claims.


In [ ]:
query_14 = """
SELECT claim_id,
       member_id,
       claim_date,
       claim_amount,
       COUNT(*) OVER (
           PARTITION BY member_id, claim_date
       ) AS same_day_claim_count,
       CASE
           WHEN COUNT(*) OVER (PARTITION BY member_id, claim_date) > 1 THEN 1
           ELSE 0
       END AS multi_claim_flag
FROM claims
ORDER BY member_id, claim_date
"""
pd.read_sql_query(query_14, con)

### Question 15
Show aging for pending claims.


In [ ]:
query_15 = """
SELECT claim_id,
       member_id,
       provider_id,
       claim_date,
       claim_amount,
       ROUND(julianday('now') - julianday(claim_date), 0) AS claim_age_days
FROM claims
WHERE claim_status = 'Pending'
ORDER BY claim_age_days DESC
"""
pd.read_sql_query(query_15, con)

## 10. Window Functions

### Question 16
Find the latest claim for each member using `ROW_NUMBER()`.


In [ ]:
query_16 = """
SELECT member_id,
       claim_id,
       claim_date,
       claim_amount
FROM (
    SELECT member_id,
           claim_id,
           claim_date,
           claim_amount,
           ROW_NUMBER() OVER (
               PARTITION BY member_id
               ORDER BY claim_date DESC
           ) AS rn
    FROM claims
) ranked_claims
WHERE rn = 1
ORDER BY member_id
"""
pd.read_sql_query(query_16, con)

### Question 17
Rank claims by amount inside each provider.


In [ ]:
query_17 = """
SELECT provider_id,
       claim_id,
       claim_amount,
       RANK() OVER (
           PARTITION BY provider_id
           ORDER BY claim_amount DESC
       ) AS claim_rank
FROM claims
ORDER BY provider_id, claim_rank
"""
pd.read_sql_query(query_17, con)

### Question 18
Get the top 2 highest claims for each provider.


In [ ]:
query_18 = """
SELECT provider_id,
       claim_id,
       claim_amount,
       claim_rank
FROM (
    SELECT provider_id,
           claim_id,
           claim_amount,
           ROW_NUMBER() OVER (
               PARTITION BY provider_id
               ORDER BY claim_amount DESC
           ) AS claim_rank
    FROM claims
) ranked_claims
WHERE claim_rank <= 2
ORDER BY provider_id, claim_rank
"""
pd.read_sql_query(query_18, con)

### Question 19
Calculate running total of claim amount by claim date.


In [ ]:
query_19 = """
SELECT claim_id,
       claim_date,
       claim_amount,
       SUM(claim_amount) OVER (
           ORDER BY claim_date
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM claims
ORDER BY claim_date
"""
pd.read_sql_query(query_19, con)

### Question 20
Calculate running total by provider.


In [ ]:
query_20 = """
SELECT provider_id,
       claim_id,
       claim_date,
       claim_amount,
       SUM(claim_amount) OVER (
           PARTITION BY provider_id
           ORDER BY claim_date
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS provider_running_total
FROM claims
ORDER BY provider_id, claim_date
"""
pd.read_sql_query(query_20, con)

### Question 21
Calculate a 3-claim moving average per provider.


In [ ]:
query_21 = """
SELECT provider_id,
       claim_id,
       claim_date,
       claim_amount,
       ROUND(
           AVG(claim_amount) OVER (
               PARTITION BY provider_id
               ORDER BY claim_date
               ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
           ),
           2
       ) AS moving_avg_3_claims
FROM claims
ORDER BY provider_id, claim_date
"""
pd.read_sql_query(query_21, con)

### Question 22
Compare each member's current claim amount with the previous claim using `LAG()`.


In [ ]:
query_22 = """
SELECT member_id,
       claim_id,
       claim_date,
       claim_amount,
       LAG(claim_amount) OVER (
           PARTITION BY member_id
           ORDER BY claim_date
       ) AS previous_claim_amount,
       claim_amount - LAG(claim_amount) OVER (
           PARTITION BY member_id
           ORDER BY claim_date
       ) AS amount_difference
FROM claims
ORDER BY member_id, claim_date
"""
pd.read_sql_query(query_22, con)

### Question 23
Show the next claim date for each member using `LEAD()`.


In [ ]:
query_23 = """
SELECT member_id,
       claim_id,
       claim_date,
       LEAD(claim_date) OVER (
           PARTITION BY member_id
           ORDER BY claim_date
       ) AS next_claim_date
FROM claims
ORDER BY member_id, claim_date
"""
pd.read_sql_query(query_23, con)

## 11. Common Table Expressions (CTEs)

### Question 24
Use a CTE to find high-value claims and then calculate provider-level total of those claims.


In [ ]:
query_24 = """
WITH high_value_claims AS (
    SELECT claim_id,
           provider_id,
           claim_amount
    FROM claims
    WHERE claim_amount > 15000
)
SELECT provider_id,
       COUNT(*) AS high_value_claim_count,
       SUM(claim_amount) AS high_value_claim_total
FROM high_value_claims
GROUP BY provider_id
ORDER BY high_value_claim_total DESC
"""
pd.read_sql_query(query_24, con)

### Question 25
Use a CTE to calculate approval percentage by provider.


In [ ]:
query_25 = """
WITH provider_summary AS (
    SELECT provider_id,
           SUM(CASE WHEN claim_status = 'Approved' THEN 1 ELSE 0 END) AS approved_claims,
           COUNT(*) AS total_claims
    FROM claims
    GROUP BY provider_id
)
SELECT provider_id,
       approved_claims,
       total_claims,
       ROUND(approved_claims * 100.0 / total_claims, 2) AS approval_percentage
FROM provider_summary
ORDER BY approval_percentage DESC, provider_id
"""
pd.read_sql_query(query_25, con)

## 12. Final Business Insights

### Key Takeaways
- UHC and Aetna are the strongest providers by claim value in this sample
- A few members have repeated activity, which is useful for utilization review
- High-value claims can be flagged for manual review
- Window functions help answer interview-style analytical questions efficiently
- This Notebook reflects a realistic healthcare reporting and analytics workflow


## 13. Next Step for GitHub

This notebook contains SQL-based analysis on healthcare claims data, covering key areas such as provider performance, claim trends, and member activity.  
`01_healthcare_claims_analysis.ipynb`
